# Raw signal vs. FeatureGraph objects, through Nori

A small, self-contained comparison: does giving Nori FeatureGraph's
constructed objects, instead of a raw window of the signal, change how
well it predicts something?

**Scope, stated up front:** one subject, one arbitrary smoothing window,
one simple chronological split. This is illustrative, not a benchmark --
see the Claim Boundary cell at the end for what this does and doesn't show.

FeatureGraph objects here come only from the hosted API -- no compiler
code runs in this notebook. The raw signal is loaded directly from the
same public dataset, independent of the API, so both sides describe the
same underlying recording.

In [ ]:
import os
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://featuregraph-api.fly.dev"
FEATUREGRAPH_API_KEY = os.environ["FEATUREGRAPH_API_KEY"]
SYNTHEFY_API_KEY = os.environ["SYNTHEFY_API_KEY"]

SUBJECT = "S01"
SMOOTH_WINDOW = 20  # arbitrary -- see the claim boundary below

## 1. The raw signal

Loaded directly from the public dataset -- this is data loading only, not
the compiler. Used here purely to build the "raw" side of the comparison.

In [ ]:
from featuregraph_smoothing_core.datasets import wearable_hr

raw = wearable_hr(subject=SUBJECT)
raw.head()

## 2. FeatureGraph objects, from the hosted API

The only way this notebook touches the compiler at all.

In [ ]:
r = requests.get(
    f"{BASE_URL}/process",
    params={"dataset": "wearable_hr", "smooth_window": SMOOTH_WINDOW},
    headers={"X-API-Key": FEATUREGRAPH_API_KEY},
    timeout=60,
)
r.raise_for_status()
result = r.json()

objects = pd.DataFrame(result["summary"])
objects = objects[objects["is_complete"]].reset_index(drop=True)
print(f"{len(objects)} complete objects")
objects.head()

## 3. Build two feature tables for the same objects

**Raw:** mean/std/min/max of the raw signal within each object's own
window -- no construction, just summary statistics of the same span.

**FeatureGraph:** the object fields already returned by `/process`.

**Target:** the *next* object's `amplitude_raw` -- a real forecasting
task ("how big is the next swing"), with no external labels and no
leakage, since the target always comes from a different row than its
own features.

In [ ]:
def raw_window_stats(row):
    window = raw["heart_rate_bpm"].iloc[int(row["start_index"]):int(row["end_index"])]
    return pd.Series({
        "raw_mean": window.mean(),
        "raw_std": window.std(),
        "raw_min": window.min(),
        "raw_max": window.max(),
    })

raw_features = objects.apply(raw_window_stats, axis=1)

fg_features = objects[[
    "duration", "amplitude_smooth", "temporal_symmetry",
    "smooth_rising_mean_rate", "smooth_falling_mean_rate",
]]

target = objects["amplitude_raw"].shift(-1)

table = pd.concat([raw_features, fg_features, target.rename("target_next_amplitude")], axis=1)
table = table.dropna().reset_index(drop=True)
print(f"{len(table)} usable rows after dropping the last object (no 'next') and any NaNs")
table.head()

## 4. A simple chronological split

First 80% as context, last 20% held out -- not shuffled, since these are
objects in time order from one recording.

In [ ]:
split = int(len(table) * 0.8)
train, test = table.iloc[:split], table.iloc[split:]

y_train = train["target_next_amplitude"].tolist()
y_test = test["target_next_amplitude"].tolist()

raw_cols = ["raw_mean", "raw_std", "raw_min", "raw_max"]
fg_cols = ["duration", "amplitude_smooth", "temporal_symmetry", "smooth_rising_mean_rate", "smooth_falling_mean_rate"]

X_train_raw = train[raw_cols].values.tolist()
X_test_raw = test[raw_cols].values.tolist()
X_train_fg = train[fg_cols].values.tolist()
X_test_fg = test[fg_cols].values.tolist()

print(f"train={len(train)}  test={len(test)}")

## 5. Three predictions: baseline, Nori-on-raw, Nori-on-FeatureGraph

The training-mean baseline is included on purpose -- the same discipline
as our own published interoperability study: a learned model that can't
beat a trivial baseline is reported plainly, not hidden.

In [ ]:
from synthefy import SynthefyNoriClient

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

baseline_pred = [np.mean(y_train)] * len(y_test)
baseline_mae = mae(y_test, baseline_pred)

client = SynthefyNoriClient(api_key=SYNTHEFY_API_KEY, model="nori-6m")

nori_raw_pred = client.predict(X_train=X_train_raw, y_train=y_train, X_test=X_test_raw)
nori_raw_mae = mae(y_test, nori_raw_pred)

nori_fg_pred = client.predict(X_train=X_train_fg, y_train=y_train, X_test=X_test_fg)
nori_fg_mae = mae(y_test, nori_fg_pred)

pd.DataFrame({
    "condition": ["training-mean baseline", "Nori on raw window stats", "Nori on FeatureGraph objects"],
    "MAE": [baseline_mae, nori_raw_mae, nori_fg_mae],
})

## Claim boundary

This notebook shows that FeatureGraph's hosted output plugs directly into
Nori with no reshaping beyond picking a target column -- that's the point
being demonstrated.

It does **not** show that:
- FeatureGraph's representation generalizes beyond this one subject;
- `smooth_window=20` is the right window for this signal (it wasn't
  chosen by any validated method -- see `featuregraph-smoothing-core`'s
  own paper for why that choice matters);
- either MAE here is good or bad in any absolute sense, with one
  train/test split and no cross-validation;
- this result would hold on a different subject, signal, or target.

For a properly controlled version of this question -- participant-held-out
folds, a frozen contract, a training-mean negative control, stated across
33 participants -- see our [PhysioNet/Nori interoperability study](https://github.com/featuregraph/featuregraph/blob/main/artifacts/studies/physionet_nori_interoperability_study.md),
which also reports its result -- including where the learned models did
*not* beat a simple baseline -- plainly.